# Titanic Dataset — Professional Data Cleaning

**Objective:** systematically profile, clean, validate, and export the Titanic passenger data using Python, pandas, and NumPy. Every transformation is tied to an observed quality issue and recorded so the workflow is reproducible and auditable.

The notebook deliberately separates **diagnosis**, **decisions**, **transformation**, and **validation**. The raw dataframe is never overwritten.

In [22]:
from pathlib import Path
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    display = print

SOURCE_PATH = Path(r"D:\OASIS\L1T3\Titanic-Dataset.csv")
OUTPUT_PATH = Path(r"D:\OASIS\L1T3\Titanic-Dataset-Cleaned.csv")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Load data and preserve the raw source

`raw` remains unchanged throughout. `df` is the working copy. This makes before/after comparisons reliable and prevents accidental mutation of the evidence.

In [23]:
raw = pd.read_csv(SOURCE_PATH)
df = raw.copy(deep=True)
print(f"Loaded {raw.shape[0]:,} rows × {raw.shape[1]} columns from {SOURCE_PATH.name}")
display(raw.head())

Loaded 891 rows × 12 columns from Titanic-Dataset.csv


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.00,1,0,A/5 21171,7.25,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.00,1,0,PC 17599,71.28,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.00,0,0,STON/O2. 3101282,7.92,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.00,1,0,113803,53.10,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.00,0,0,373450,8.05,NaN,S


## 2. Data quality report

The report measures missingness, inferred dtype, uniqueness, duplicate rows, type conformity, and domain/range anomalies. Domain rules are based on the Titanic schema: binary survival, passenger class 1–3, non-negative ages/fares/family counts, and known category codes.

In [24]:
expected_types = {
    "PassengerId": "identifier", "Survived": "binary", "Pclass": "category",
    "Name": "text", "Sex": "category", "Age": "numeric", "SibSp": "integer",
    "Parch": "integer", "Ticket": "identifier", "Fare": "numeric",
    "Cabin": "category", "Embarked": "category"
}

def dtype_is_accurate(series, expected):
    if expected in {"numeric", "integer", "binary"}:
        return pd.api.types.is_numeric_dtype(series)
    return pd.api.types.is_string_dtype(series) or pd.api.types.is_object_dtype(series) or isinstance(series.dtype, pd.CategoricalDtype)

def anomaly_mask(frame, column):
    rules = {
        "PassengerId": ~frame["PassengerId"].between(1, 10_000),
        "Survived": ~frame["Survived"].isin([0, 1]),
        "Pclass": ~frame["Pclass"].isin([1, 2, 3]),
        "Sex": ~frame["Sex"].astype("string").str.strip().str.lower().isin(["male", "female"]),
        "Age": frame["Age"].notna() & ~frame["Age"].between(0, 100),
        "SibSp": frame["SibSp"] < 0,
        "Parch": frame["Parch"] < 0,
        "Fare": frame["Fare"].notna() & (frame["Fare"] < 0),
        "Embarked": frame["Embarked"].notna() & ~frame["Embarked"].astype("string").str.strip().str.upper().isin(["C", "Q", "S"]),
    }
    return rules.get(column, pd.Series(False, index=frame.index))

quality_before = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "expected_role": pd.Series(expected_types),
    "dtype_accurate": [dtype_is_accurate(raw[c], expected_types[c]) for c in raw.columns],
    "null_count": raw.isna().sum(),
    "null_pct": raw.isna().mean().mul(100).round(2),
    "unique_count": raw.nunique(dropna=False),
    "range_or_domain_anomalies": [int(anomaly_mask(raw, c).sum()) for c in raw.columns],
})

print(f"Exact duplicate rows: {raw.duplicated().sum():,}")
display(quality_before)
display(raw.select_dtypes(include=np.number).agg(["min", "max", "mean", "median"]).T)

Exact duplicate rows: 0


,dtype,expected_role,dtype_accurate,null_count,null_pct,unique_count,range_or_domain_anomalies
PassengerId,int64,identifier,False,0,0.00,891,0
Survived,int64,binary,True,0,0.00,2,0
Pclass,int64,category,False,0,0.00,3,0
Name,str,text,True,0,0.00,891,0
Sex,str,category,True,0,0.00,2,0
Age,float64,numeric,True,177,19.87,89,0
SibSp,int64,integer,True,0,0.00,7,0
Parch,int64,integer,True,0,0.00,7,0
Ticket,str,identifier,True,0,0.00,681,0
Fare,float64,numeric,True,0,0.00,248,0


,min,max,mean,median
PassengerId,1.00,891.00,446.00,446.00
Survived,0.00,1.00,0.38,0.00
Pclass,1.00,3.00,2.31,3.00
Age,0.42,80.00,29.70,28.00
SibSp,0.00,8.00,0.52,0.00
Parch,0.00,6.00,0.38,0.00
Fare,0.00,512.33,32.20,14.45


### Quality findings

- **Missingness:** `Age` has 177 nulls, `Cabin` 687, and `Embarked` 2.
- **Duplicates:** exact full-row duplicates are checked separately. A repeated ticket or cabin is not automatically a duplicate because passengers can travel together.
- **Type issues:** the imported numeric columns are machine-readable, but semantic types need correction: identifiers should be strings; `Survived` should be boolean; low-cardinality fields should be categorical.
- **Range/domain anomalies:** impossible values are tested explicitly. High fares and unusual family sizes are not labelled invalid merely because they are rare.
- **Formatting:** text fields are normalized with whitespace trimming and canonical category mappings, even if the current file already appears consistent. This makes the pipeline robust to variants such as `M`, `male`, or ` Male `.

## 3. Duplicate handling

Only **exact full-row duplicates** are removed. `PassengerId`, `Ticket`, and `Cabin` are not used alone as duplicate keys: shared bookings/cabins are legitimate, while `PassengerId` is validated independently for uniqueness.

In [25]:
duplicate_count_before = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
duplicates_removed = duplicate_count_before - int(df.duplicated().sum())
print(f"Exact duplicate rows removed: {duplicates_removed:,}")
print(f"Duplicate PassengerId values remaining: {df['PassengerId'].duplicated().sum():,}")

Exact duplicate rows removed: 0
Duplicate PassengerId values remaining: 0


## 4. Standardise text and categorical values

- Strip leading/trailing and repeated internal whitespace from free text.
- Map common sex variants to `Male` / `Female`.
- Map embarkation names and case variants to the canonical port codes `C`, `Q`, `S`.
- Keep ticket and cabin as strings because their letters, punctuation, and leading structure carry meaning.

In [26]:
text_columns = ["Name", "Sex", "Ticket", "Cabin", "Embarked"]
for col in text_columns:
    df[col] = df[col].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)

sex_map = {"m": "Male", "male": "Male", "f": "Female", "female": "Female"}
df["Sex"] = df["Sex"].str.lower().map(sex_map).fillna(df["Sex"].str.title())

embarked_map = {
    "c": "C", "cherbourg": "C",
    "q": "Q", "queenstown": "Q",
    "s": "S", "southampton": "S",
}
df["Embarked"] = df["Embarked"].str.lower().map(embarked_map)
print("Sex values:", sorted(df["Sex"].dropna().unique()))
print("Embarked values before imputation:", sorted(df["Embarked"].dropna().unique()))

Sex values: ['Female', 'Male']
Embarked values before imputation: ['C', 'Q', 'S']


## 5. Missing-data decisions

| Column | Strategy | Justification |
|---|---|---|
| Age | Median within extracted title and passenger class; fallback to class median, then global median | Age is skewed and strongly related to social title/class. Group medians preserve those relationships and resist extreme ages better than the mean. |
| Cabin | Fill with `Unknown` | About 77% is missing. Deleting rows would discard most of the dataset, and guessing a cabin would create false precision. Missingness itself may be informative. |
| Embarked | Mode imputation | Only two values are missing and the field is categorical; the mode is a simple, transparent choice with negligible impact. |
| Other columns | No imputation | They have no missing values. Forward fill is inappropriate because rows are independent passengers, not a time series. Row deletion is unnecessary. |

In [27]:
missing_before = df.isna().sum().copy()

df["Title"] = (
    df["Name"].str.extract(r",\s*([^.]*)\.", expand=False).str.strip()
      .replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
)
rare_titles = ~df["Title"].isin(["Mr", "Mrs", "Miss", "Master"])
df.loc[rare_titles & df["Title"].notna(), "Title"] = "Rare"

title_class_medians = df.groupby(["Title", "Pclass"], observed=True)["Age"].transform("median")
class_medians = df.groupby("Pclass", observed=True)["Age"].transform("median")
df["Age"] = df["Age"].fillna(title_class_medians).fillna(class_medians).fillna(df["Age"].median())

df["Cabin"] = df["Cabin"].fillna("Unknown")
embarked_mode = df["Embarked"].mode(dropna=True).iat[0]
df["Embarked"] = df["Embarked"].fillna(embarked_mode)

imputation_log = pd.DataFrame({
    "missing_before": missing_before,
    "missing_after": df.isna().sum(),
})
imputation_log["values_imputed"] = imputation_log["missing_before"] - imputation_log["missing_after"]
display(imputation_log.query("missing_before > 0"))

,missing_before,missing_after,values_imputed
Age,177.00,0,177.00
Cabin,687.00,0,687.00
Embarked,2.00,0,2.00


## 6. Outlier detection and decisions

The 1.5×IQR rule is applied to continuous/count measures: `Age`, `SibSp`, `Parch`, and `Fare`. It is **not** applied to IDs, binary survival, or passenger class because numerical distance is not meaningful for those fields.

Decision: **retain all IQR outliers and add transparent flag columns.** These values are plausible for the domain: infants and older passengers are valid ages; large family groups are possible; and first-class group bookings can have very high fares. Removing or capping them without evidence of error would erase real passengers and distort survival analysis. The flags let downstream analysts run sensitivity checks.

In [28]:
outlier_rows = []
for col in ["Age", "SibSp", "Parch", "Fare"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flag_col = f"{col}_IQR_Outlier"
    df[flag_col] = ~df[col].between(lower, upper)
    outlier_rows.append({
        "column": col, "q1": q1, "q3": q3, "iqr": iqr,
        "lower_fence": lower, "upper_fence": upper,
        "outlier_count": int(df[flag_col].sum()), "decision": "Retain and flag"
    })

outlier_report = pd.DataFrame(outlier_rows).set_index("column")
display(outlier_report)

,q1,q3,iqr,lower_fence,upper_fence,outlier_count,decision
column,,,,,,,
Age,21.00,36.75,15.75,-2.62,60.38,22,Retain and flag
SibSp,0.00,1.00,1.00,-1.50,2.50,46,Retain and flag
Parch,0.00,0.00,0.00,0.00,0.00,213,Retain and flag
Fare,7.91,31.00,23.09,-26.72,65.63,116,Retain and flag


## 7. Correct semantic data types

- IDs (`PassengerId`, `Ticket`) → string, preventing accidental arithmetic.
- `Survived` and outlier indicators → boolean.
- `Pclass`, `Sex`, `Cabin`, `Embarked`, `Title` → categorical.
- `Age` and `Fare` → float; family counts → compact nullable integers.

CSV files do not store pandas dtype metadata, so these types are enforced in memory and documented here. On a future reload, the same schema should be applied.

In [29]:
df["PassengerId"] = df["PassengerId"].astype("Int64").astype("string")
df["Ticket"] = df["Ticket"].astype("string")
df["Survived"] = df["Survived"].astype("boolean")
df["Pclass"] = df["Pclass"].astype("category")
for col in ["Sex", "Cabin", "Embarked", "Title"]:
    df[col] = df[col].astype("category")
for col in ["SibSp", "Parch"]:
    df[col] = df[col].astype("Int64")
for col in ["Age", "Fare"]:
    df[col] = df[col].astype("float64")
for col in [c for c in df.columns if c.endswith("_IQR_Outlier")]:
    df[col] = df[col].astype("boolean")

display(df.dtypes.to_frame("clean_dtype"))

,clean_dtype
PassengerId,string
Survived,boolean
Pclass,category
Name,string
Sex,category
Age,float64
SibSp,Int64
Parch,Int64
Ticket,string
Fare,float64


## 8. Before vs. after summary

`dtype_accuracy` is the proportion of the 12 original fields that match the intended semantic schema. The cleaned dataset also contains one derived `Title` feature and four outlier flags; these are validated separately.

In [30]:
def clean_dtype_accurate(column):
    checks = {
        "PassengerId": pd.api.types.is_string_dtype(df[column]),
        "Survived": str(df[column].dtype) == "boolean",
        "Pclass": isinstance(df[column].dtype, pd.CategoricalDtype),
        "Name": pd.api.types.is_string_dtype(df[column]),
        "Sex": isinstance(df[column].dtype, pd.CategoricalDtype),
        "Age": pd.api.types.is_float_dtype(df[column]),
        "SibSp": str(df[column].dtype) == "Int64",
        "Parch": str(df[column].dtype) == "Int64",
        "Ticket": pd.api.types.is_string_dtype(df[column]),
        "Fare": pd.api.types.is_float_dtype(df[column]),
        "Cabin": isinstance(df[column].dtype, pd.CategoricalDtype),
        "Embarked": isinstance(df[column].dtype, pd.CategoricalDtype),
    }
    return checks[column]

before_dtype_accuracy = quality_before["dtype_accurate"].mean()
after_dtype_accuracy = np.mean([clean_dtype_accurate(c) for c in raw.columns])

summary = pd.DataFrame({
    "Before": [len(raw), int(raw.isna().sum().sum()), int(raw.duplicated().sum()), f"{before_dtype_accuracy:.1%}"],
    "After": [len(df), int(df.isna().sum().sum()), int(df.duplicated().sum()), f"{after_dtype_accuracy:.1%}"],
}, index=["Row count", "Total null cells", "Exact duplicate rows", "Dtype accuracy"])
display(summary)

,Before,After
Row count,891,891
Total null cells,866,0
Exact duplicate rows,0,0
Dtype accuracy,83.3%,100.0%


## 9. Final validation and export

Assertions turn quality expectations into executable tests. Export occurs only if every required invariant passes: no nulls, no exact duplicates, unique IDs, canonical categories, valid ranges, and accurate semantic dtypes.

In [31]:
assert df.isna().sum().sum() == 0, "Null values remain"
assert df.duplicated().sum() == 0, "Exact duplicates remain"
assert df["PassengerId"].is_unique, "PassengerId must be unique"
assert set(df["Survived"].dropna().unique()).issubset({True, False})
assert set(df["Pclass"].astype(int).unique()).issubset({1, 2, 3})
assert set(df["Sex"].astype(str).unique()).issubset({"Male", "Female"})
assert set(df["Embarked"].astype(str).unique()).issubset({"C", "Q", "S"})
assert df["Age"].between(0, 100).all()
assert (df[["SibSp", "Parch", "Fare"]] >= 0).all().all()
assert after_dtype_accuracy == 1.0

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)

print(f"Validation passed: {len(df):,} clean rows, {df.shape[1]} columns")
print(f"Saved cleaned dataset to: {OUTPUT_PATH}")
display(df.head(20))

Validation passed: 891 clean rows, 17 columns
Saved cleaned dataset to: D:\OASIS\L1T3\Titanic-Dataset-Cleaned.csv


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,Age_IQR_Outlier,SibSp_IQR_Outlier,Parch_IQR_Outlier,Fare_IQR_Outlier
0,1,False,3,"Braund, Mr. Owen Harris",Male,22.00,1,0,A/5 21171,7.25,Unknown,S,Mr,False,False,False,False
1,2,True,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Female,38.00,1,0,PC 17599,71.28,C85,C,Mrs,False,False,False,True
2,3,True,3,"Heikkinen, Miss. Laina",Female,26.00,0,0,STON/O2. 3101282,7.92,Unknown,S,Miss,False,False,False,False
3,4,True,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Female,35.00,1,0,113803,53.10,C123,S,Mrs,False,False,False,False
4,5,False,3,"Allen, Mr. William Henry",Male,35.00,0,0,373450,8.05,Unknown,S,Mr,False,False,False,False
5,6,False,3,"Moran, Mr. James",Male,26.00,0,0,330877,8.46,Unknown,Q,Mr,False,False,False,False
6,7,False,1,"McCarthy, Mr. Timothy J",Male,54.00,0,0,17463,51.86,E46,S,Mr,False,False,False,False
7,8,False,3,"Palsson, Master. Gosta Leonard",Male,2.00,3,1,349909,21.07,Unknown,S,Master,False,True,True,False
8,9,True,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",Female,27.00,0,2,347742,11.13,Unknown,S,Mrs,False,False,True,False
9,10,True,2,"Nasser, Mrs. Nicholas (Adele Achem)",Female,14.00,1,0,237736,30.07,Unknown,C,Mrs,False,False,False,False


## Cleaning decision log

| Step | Decision | Result |
|---|---|---|
| Raw preservation | Work on a deep copy | Before/after evidence remains intact |
| Duplicates | Remove exact full-row duplicates only | Count reported in output above |
| Formatting | Trim/collapse whitespace; canonicalize `Sex` and `Embarked` | Stable category labels |
| Age missingness | Hierarchical median imputation by title/class | All ages populated without mean sensitivity |
| Cabin missingness | Explicit `Unknown` category | Avoided deleting most rows or inventing cabins |
| Embarked missingness | Mode imputation | Filled two categorical gaps transparently |
| Outliers | IQR detection; retain and flag | Preserved plausible rare passengers for sensitivity analysis |
| Types | Apply semantic pandas dtypes | Prevented arithmetic on IDs and clarified categories/booleans |
| Validation | Executable assertions before export | CSV produced only after all checks passed |